In [1]:
import xarray as xr
from xarray import DataTree
from pathlib import Path
import pandas as pd
from dask.diagnostics import ProgressBar
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import xclim

import cartopy.crs as ccrs
import valenspy as vp
from valenspy.diagnostic.visualizations import _add_features
from valenspy.diagnostic.functions import root_mean_square_error
from valenspy._utilities import CORDEX_VARIABLES
import xesmf

In [2]:
manager = vp.InputManager(machine="hortense")
variables = ["tas","pr"]

In [3]:
ds_mar = xr.open_mfdataset("/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc44757_Nicolas/remap_mar/MAR-ERA5/*.nc")
from valenspy.input import INPUT_CONVERTORS
ds_mar = INPUT_CONVERTORS["MAR"].convert_input(ds_mar)
#Keep only the variables of interest
ds_mar = ds_mar[variables]

# Observational data
## CLIMATE_GRID (Regridded data)

ds_ref = manager.load_data("CLIMATE_GRID", variables, path_identifiers=["regridded"])

Variable metadata is missing or incorrect
The file is NOT ValEnsPy CF compliant.
78.57% of the variables are ValEnsPy CF compliant
ValEnsPy CF compliant: ['tas', 'huss', 'hurs', 'uas', 'vas', 'rsds', 'clt', 'tasmax', 'tasmin', 'prsn', 'pr']
Unknown to ValEnsPy: ['TIME_bnds', 'ZUVLEV_bnds', 'RHZmin']
File paths found:
/dodrio/scratch/projects/2022_200/external/climate_grid/regridded/latlon_5km/PRECIP_QUANTITY_CLIMATE_GRID_1951_2024_daily_latlon_5km.nc
/dodrio/scratch/projects/2022_200/external/climate_grid/regridded/latlon_5km/TEMP_AVG_CLIMATE_GRID_1954_2024_daily_latlon_5km.nc
The file is ValEnsPy CF compliant.
100.00% of the variables are ValEnsPy CF compliant
ValEnsPy CF compliant: ['pr', 'tas']


In [4]:
data_dict = {
    # "RCM/ERA5/ALARO1_SFX": ds_alaro,
    # "RCM/ERA5/CCLM6-0-1-URB-ESG": ds_cclm, 
    "RCM/ERA5/MAR": ds_mar,
    "obs/CLIMATE_GRID": ds_ref
}

In [5]:
dt = DataTree.from_dict(data_dict)

In [6]:
import xclim

In [7]:
from valenspy.processing import convert_units_to

In [8]:
dt

<xarray.DataTree>
Group: /
├── Group: /obs
│   └── Group: /obs/CLIMATE_GRID
│           Dimensions:  (time: 27029, lon: 75, lat: 70)
│           Coordinates:
│             * time     (time) datetime64[ns] 216kB 1951-01-01 1951-01-02 ... 2024-12-31
│             * lon      (lon) float64 600B 2.0 2.07 2.14 2.21 2.28 ... 6.97 7.04 7.11 7.18
│             * lat      (lat) float64 560B 49.0 49.05 49.09 49.13 ... 52.02 52.06 52.1
│           Data variables:
│               pr       (time, lat, lon) float64 1GB dask.array<chunksize=(3195, 70, 75), meta=np.ndarray>
│               tas      (time, lat, lon) float64 1GB dask.array<chunksize=(2881, 70, 75), meta=np.ndarray>
│           Attributes: (12/15)
│               CDI:                 Climate Data Interface version 2.4.1 (https://mpimet...
│               Conventions:         CF-1.6
│               creation_date:       28-03-2025
│               creators:            Ghilain N., Van Schaeybroeck B., Vanderkelen I.
│               contact:             inne.vanderkelen@meteo.be
│               version:             1.1
│               ...                  ...
│               CDO:                 Climate Data Operators version 2.4.1 (https://mpimet...
│               freq:                day
│               spatial_resolution:  0.07° x 0.045° (~5km)
│               region:              Belgium
│               dataset:             CLIMATE_GRID
│               path_identifiers:    ['regridded']
└── Group: /RCM
    └── Group: /RCM/ERA5
        └── Group: /RCM/ERA5/MAR
                Dimensions:  (time: 16071, lat: 74, lon: 132)
                Coordinates:
                  * time     (time) datetime64[ns] 129kB 1980-01-01T12:00:00 ... 2023-12-31T1...
                  * lon      (lon) float64 1kB 1.46 1.51 1.56 1.61 1.66 ... 7.86 7.91 7.96 8.01
                  * lat      (lat) float64 592B 52.45 52.4 52.35 52.3 ... 48.95 48.9 48.85 48.8
                    ZTQLEV   float32 4B 2.0
                    ZUVLEV   float32 4B 2.0
                Data variables:
                    tas      (time, lat, lon) float32 628MB dask.array<chunksize=(1, 74, 132), meta=np.ndarray>
                    pr       (time, lat, lon) float32 628MB dask.array<chunksize=(1, 74, 132), meta=np.ndarray>
                Attributes:
                    CDI:          Climate Data Interface version 2.4.4 (https://mpimet.mpg.de...
                    Conventions:  CF-1.6
                    institute:    University of Liège (Belgium)
                    contact:      xavierfettweis@uliege.be
                    model:        regional climate model MARv3.13
                    NCO:          netCDF Operators version 5.2.4 (Homepage = http://nco.sf.ne...
                    history:      Thu Mar 20 15:11:35 2025: cdo remapbil,gridoutput -setgrid,...
                    frequency:    day
                    CDO:          Climate Data Operators version 2.4.4 (https://mpimet.mpg.de...
                    dataset:      MAR
                    freq:         day
                    region:       Belgium

In [9]:
dt_s = convert_units_to(dt, "tas", "C")

In [10]:
dt.obs.CLIMATE_GRID.tas.attrs

{'long_name': 'Near-Surface Air Temperature',
 'units': 'K',
 'description': np.float64(nan),
 'units_metadata': 'temperature: unknown',
 'standard_name': 'air_temperature',
 'original_name': 'TEMP_AVG',
 'original_units': 'degC',
 'freq': 'day',
 'spatial_resolution': '0.07° x 0.045° (~5km)',
 'region': 'Belgium',
 'dataset': 'CLIMATE_GRID',
 'path_identifiers': ['regridded']}

In [11]:
dt_s.obs.CLIMATE_GRID.tas.attrs

{'long_name': 'Near-Surface Air Temperature',
 'units': '°C',
 'description': np.float64(nan),
 'units_metadata': 'temperature: unknown',
 'standard_name': 'air_temperature',
 'original_name': 'TEMP_AVG',
 'original_units': 'degC',
 'freq': 'day',
 'spatial_resolution': '0.07° x 0.045° (~5km)',
 'region': 'Belgium',
 'dataset': 'CLIMATE_GRID',
 'path_identifiers': ['regridded']}

In [12]:
xr.__version__

'2025.3.1'

In [13]:
from valenspy.processing import xclim_indicator

xclim_indicator(dt, xclim.indicators.cf.tnn, vars="tas", freq="YS")

TypeError: xclim_indicator.<locals>.xclim_indicator_ds() takes 3 positional arguments but 4 were given

In [14]:
from xclim.core.indicator import Indicator
def xclim_indicator_ds(ds: xr.Dataset, indicator: Indicator,  vars: str | list, **kwargs) -> xr.Dataset:
    """
    Wrapper function to apply the indicator to a single dataset.
    """
    if isinstance(vars, str):
        if vars in ds:
            return indicator(ds[vars], **kwargs).to_dataset()
    elif isinstance(vars, list): #Order is important!
        data_arrays = [ds[var] for var in vars]
        if vars in ds:
            return indicator(*data_arrays, **kwargs).to_dataset()

In [15]:
dt.map_over_datasets(
        xclim_indicator_ds,
        xclim.indicators.cf.tnn,
        "tas",
        kwargs={"freq":"YS"}
    )

/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/conda_envs/valenspy_datatree/lib/python3.11/site-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/conda_envs/valenspy_datatree/lib/python3.11/site-packages/xclim/core/cfchecks.py:77: UserWarning: Variable has a non-conforming cell_methods: Got `TIME: mean`, which do not include the expected `time: minimum`.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])


<xarray.DataTree>
Group: /
├── Group: /obs
│   └── Group: /obs/CLIMATE_GRID
│           Dimensions:  (lon: 75, lat: 70, time: 74)
│           Coordinates:
│             * lon      (lon) float64 600B 2.0 2.07 2.14 2.21 2.28 ... 6.97 7.04 7.11 7.18
│             * lat      (lat) float64 560B 49.0 49.05 49.09 49.13 ... 52.02 52.06 52.1
│             * time     (time) datetime64[ns] 592B 1951-01-01 1952-01-01 ... 2024-01-01
│           Data variables:
│               tnn      (time, lat, lon) float64 3MB dask.array<chunksize=(1, 70, 75), meta=np.ndarray>
└── Group: /RCM
    └── Group: /RCM/ERA5
        └── Group: /RCM/ERA5/MAR
                Dimensions:  (lon: 132, lat: 74, time: 44)
                Coordinates:
                  * lon      (lon) float64 1kB 1.46 1.51 1.56 1.61 1.66 ... 7.86 7.91 7.96 8.01
                  * lat      (lat) float64 592B 52.45 52.4 52.35 52.3 ... 48.95 48.9 48.85 48.8
                    ZTQLEV   float32 4B 2.0
                    ZUVLEV   float32 4B 2.0
                  * time     (time) datetime64[ns] 352B 1980-01-01 1981-01-01 ... 2023-01-01
                Data variables:
                    tnn      (time, lat, lon) float32 2MB dask.array<chunksize=(1, 74, 132), meta=np.ndarray>

In [ ]:
from xclim.testing import open_dataset

In [ ]:
ds = open_dataset("ERA5/daily_surface_cancities_1990-1993.nc")

In [ ]:
ds

In [ ]:
import xarray
xarray.__version__

'2025.3.1'

In [28]:
xclim.indicators.cf.tnn(dt.obs.CLIMATE_GRID.tas, freq="YS")

AttributeError: module 'xarray.core' has no attribute 'merge'

In [29]:
xclim.__version__

'0.54.0'